# 34 · Adaptive Mixture Boundary: Learn α(x)

This notebook upgrades Notebook 33 by replacing the fixed mixture weight with an **adaptive gate**.

Previous notebooks suggested:
- a unified boundary captures most of the zeta–GUE signal
- specialist boundaries add local corrections in some regimes
- a fixed mixture weight helps at some scales but not others
- the right picture is likely **shared core + condition-dependent correction strength**

Now we ask:

> Can we learn a data-dependent mixture weight α(x) that decides when to trust the unified core and when to lean on the specialist correction?

## Model families

We compare four strategies:
- **Unified core**
- **Specialist mixture**
- **Fixed mixture** with constant α
- **Adaptive mixture** with learned α(x)

## Gate inputs

We learn α(x) from simple local statistics:
- entropy
- unevenness
- ratio_mean

using a logistic gate:
\[
\alpha(x) = \sigma(w \cdot s(x) + b)
\]

## Condition systems

We test:
- **entropy**: low entropy + high entropy
- **unevenness**: low unevenness + high unevenness

## Tasks

For each system, we compare:
- **zeta vs GUE**
- **zeta vs Poisson**

## Main outputs

- adaptive-vs-fixed-vs-unified-vs-specialist phase-gap bars
- adaptive-gain heatmaps
- overlap comparison curves with bootstrap intervals
- learned α summaries across scales

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and feature builder

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

## RBF boundary helpers

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def fit_rbf_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    Xtr_std, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)

    return {
        "mean": mean,
        "std": std,
        "prototypes": prototypes,
        "gamma": gamma,
        "w": w,
    }

def boundary_scores(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    Xte_std = (X_test - model["mean"]) / model["std"]
    R_test = rbf_features(Xte_std, model["prototypes"], model["gamma"])
    scores = decision_function_logistic(R_test, model["w"])
    probs = predict_proba_logistic(R_test, model["w"])
    return y_test, scores, probs

def evaluate_scores(y_test, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_test))
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]
    overlap = overlap_coefficient_from_hist(scores0, scores1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

def evaluate_rbf_boundary(model, X0_test, X1_test):
    y_test, scores, probs = boundary_scores(model, X0_test, X1_test)
    return evaluate_scores(y_test, scores, probs)

## Fixed and adaptive mixture models

In [ ]:
def combine_probs(p_unified, p_specialist, alpha):
    return (1.0 - alpha) * p_unified + alpha * p_specialist

def fit_adaptive_gate(gating_features, y_true, p_unified, p_specialist, lr=0.1, n_steps=1500, reg=1e-4):
    Z = np.column_stack([np.ones(len(gating_features)), gating_features])
    theta = np.zeros(Z.shape[1])

    for _ in range(n_steps):
        alpha = sigmoid(Z @ theta)
        p_mix = combine_probs(p_unified, p_specialist, alpha)
        p_mix = np.clip(p_mix, 1e-6, 1 - 1e-6)

        dL_dp = -(y_true / p_mix) + (1 - y_true) / (1 - p_mix)
        dp_dalpha = p_specialist - p_unified
        dL_dalpha = dL_dp * dp_dalpha
        dalpha_dz = alpha * (1 - alpha)
        grad = Z.T @ (dL_dalpha * dalpha_dz) / len(Z)
        grad[1:] += reg * theta[1:]
        theta -= lr * grad

    return theta

def eval_mixture_from_alpha(y_true, p_unified, p_specialist, alpha):
    p_mix = np.clip(combine_probs(p_unified, p_specialist, alpha), 1e-8, 1 - 1e-8)
    scores_mix = np.log(p_mix / (1 - p_mix))
    return evaluate_scores(y_true, scores_mix, p_mix)

def evaluate_all_models(X0_low, X1_low, X0_high, X1_high, fixed_alpha=0.5):
    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    unified = fit_rbf_boundary(np.vstack([X0_low, X0_high]), np.vstack([X1_low, X1_high]))
    spec_low = fit_rbf_boundary(X0_low, X1_low)
    spec_high = fit_rbf_boundary(X0_high, X1_high)

    y_low, _, pu_low = boundary_scores(unified, X0_low, X1_low)
    _, _, ps_low = boundary_scores(spec_low, X0_low, X1_low)
    gate_low = np.vstack([X0_low[:, :3], X1_low[:, :3]])
    theta_low = fit_adaptive_gate(gate_low, y_low, pu_low, ps_low)
    alpha_low = sigmoid(np.column_stack([np.ones(len(gate_low)), gate_low]) @ theta_low)

    y_high, _, pu_high = boundary_scores(unified, X0_high, X1_high)
    _, _, ps_high = boundary_scores(spec_high, X0_high, X1_high)
    gate_high = np.vstack([X0_high[:, :3], X1_high[:, :3]])
    theta_high = fit_adaptive_gate(gate_high, y_high, pu_high, ps_high)
    alpha_high = sigmoid(np.column_stack([np.ones(len(gate_high)), gate_high]) @ theta_high)

    unified_low = eval_mixture_from_alpha(y_low, pu_low, pu_low, np.zeros_like(pu_low))
    unified_high = eval_mixture_from_alpha(y_high, pu_high, pu_high, np.zeros_like(pu_high))
    specialist_low = eval_mixture_from_alpha(y_low, ps_low, ps_low, np.zeros_like(ps_low))
    specialist_high = eval_mixture_from_alpha(y_high, ps_high, ps_high, np.zeros_like(ps_high))

    fixed_low = eval_mixture_from_alpha(y_low, pu_low, ps_low, fixed_alpha * np.ones_like(pu_low))
    fixed_high = eval_mixture_from_alpha(y_high, pu_high, ps_high, fixed_alpha * np.ones_like(pu_high))
    adaptive_low = eval_mixture_from_alpha(y_low, pu_low, ps_low, alpha_low)
    adaptive_high = eval_mixture_from_alpha(y_high, pu_high, ps_high, alpha_high)

    return {
        "unified_low": unified_low,
        "unified_high": unified_high,
        "unified_mixed": {
            "accuracy": float(np.mean([unified_low["accuracy"], unified_high["accuracy"]])),
            "overlap": float(np.mean([unified_low["overlap"], unified_high["overlap"]])),
        },
        "specialist_low": specialist_low,
        "specialist_high": specialist_high,
        "specialist_mixed": {
            "accuracy": float(np.mean([specialist_low["accuracy"], specialist_high["accuracy"]])),
            "overlap": float(np.mean([specialist_low["overlap"], specialist_high["overlap"]])),
        },
        "fixed_low": fixed_low,
        "fixed_high": fixed_high,
        "fixed_mixed": {
            "accuracy": float(np.mean([fixed_low["accuracy"], fixed_high["accuracy"]])),
            "overlap": float(np.mean([fixed_low["overlap"], fixed_high["overlap"]])),
        },
        "adaptive_low": adaptive_low,
        "adaptive_high": adaptive_high,
        "adaptive_mixed": {
            "accuracy": float(np.mean([adaptive_low["accuracy"], adaptive_high["accuracy"]])),
            "overlap": float(np.mean([adaptive_low["overlap"], adaptive_high["overlap"]])),
        },
        "alpha_low_mean": float(np.mean(alpha_low)),
        "alpha_high_mean": float(np.mean(alpha_high)),
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def summarize(vals):
    vals = np.array(vals)
    return {
        "mean": float(vals.mean()),
        "q025": float(np.quantile(vals, 0.025)),
        "q975": float(np.quantile(vals, 0.975)),
    }

def bootstrap_adaptive_mixture(X0_low, X1_low, X0_high, X1_high, fixed_alpha=0.5, n_boot=20, seed=9423):
    boot_rng = np.random.default_rng(seed)
    keys = [
        "unified_low","unified_high","unified_mixed",
        "specialist_low","specialist_high","specialist_mixed",
        "fixed_low","fixed_high","fixed_mixed",
        "adaptive_low","adaptive_high","adaptive_mixed",
    ]
    store = {k: [] for k in keys}
    alpha_low_vals = []
    alpha_high_vals = []

    m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high))
    X0_low = X0_low[:m]
    X1_low = X1_low[:m]
    X0_high = X0_high[:m]
    X1_high = X1_high[:m]

    for _ in range(n_boot):
        A0 = bootstrap_rows(X0_low, boot_rng)
        A1 = bootstrap_rows(X1_low, boot_rng)
        B0 = bootstrap_rows(X0_high, boot_rng)
        B1 = bootstrap_rows(X1_high, boot_rng)

        out = evaluate_all_models(A0, A1, B0, B1, fixed_alpha=fixed_alpha)
        for k in keys:
            store[k].append(out[k]["overlap"])
        alpha_low_vals.append(out["alpha_low_mean"])
        alpha_high_vals.append(out["alpha_high_mean"])

    result = {k: summarize(v) for k, v in store.items()}
    result["alpha_low"] = summarize(alpha_low_vals)
    result["alpha_high"] = summarize(alpha_high_vals)
    return result

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 120
height_block = (0, 400)
n_boot = 20
fixed_alpha = 0.5

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}

## Base gap slices

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]

len(zeta_base_gaps), len(poisson_base_gaps), len(gue_base_gaps)

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=120):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}

for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            conditioned[k][("zeta", cond)] = get_conditioned_features(zeta_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("poisson", cond)] = get_conditioned_features(poisson_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("gue", cond)] = get_conditioned_features(gue_base_gaps, cond, k=k, feature_family=feature_family, n=sample_size)

{k: {key: len(val) for key, val in conditioned[k].items()} for k in window_sizes}

## Main adaptive-mixture sweep

In [ ]:
adapt_results = []

for system_name, (cond_lo, cond_hi) in systems.items():
    for k in window_sizes:
        z_low = conditioned[k][("zeta", cond_lo)]
        z_high = conditioned[k][("zeta", cond_hi)]
        g_low = conditioned[k][("gue", cond_lo)]
        g_high = conditioned[k][("gue", cond_hi)]
        p_low = conditioned[k][("poisson", cond_lo)]
        p_high = conditioned[k][("poisson", cond_hi)]

        m = min(len(z_low), len(z_high), len(g_low), len(g_high), len(p_low), len(p_high), sample_size)
        if m < 40:
            continue

        zg = bootstrap_adaptive_mixture(z_low[:m], g_low[:m], z_high[:m], g_high[:m], fixed_alpha=fixed_alpha, n_boot=n_boot, seed=16000 + 10 * k + len(system_name))
        zp = bootstrap_adaptive_mixture(z_low[:m], p_low[:m], z_high[:m], p_high[:m], fixed_alpha=fixed_alpha, n_boot=n_boot, seed=17000 + 10 * k + len(system_name))

        row = {"system": system_name, "k": k, "sample_used": m}
        for prefix, out in [("zg", zg), ("zp", zp)]:
            for key, val in out.items():
                row[f"{prefix}_{key}_mean"] = val["mean"]
                row[f"{prefix}_{key}_q025"] = val["q025"]
                row[f"{prefix}_{key}_q975"] = val["q975"]

        for name in ["unified_low","unified_high","unified_mixed","specialist_low","specialist_high","specialist_mixed","fixed_low","fixed_high","fixed_mixed","adaptive_low","adaptive_high","adaptive_mixed"]:
            row[f"phase_gap_{name}"] = row[f"zg_{name}_mean"] - row[f"zp_{name}_mean"]

        row["alpha_low_mean"] = row["zg_alpha_low_mean"]
        row["alpha_high_mean"] = row["zg_alpha_high_mean"]

        adapt_results.append(row)

len(adapt_results)

## Quick diagnostic

In [ ]:
sorted(set(r["system"] for r in adapt_results)), sorted(set(r["k"] for r in adapt_results))

## Adaptive vs fixed vs unified vs specialist phase-gap bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in adapt_results if r["system"] == system_name]
    x = np.arange(len(window_sizes))
    width = 0.08

    metrics = [
        ("phase_gap_unified_mixed", "u-mixed"),
        ("phase_gap_specialist_mixed", "s-mixed"),
        ("phase_gap_fixed_mixed", "f-mixed"),
        ("phase_gap_adaptive_mixed", "a-mixed"),
        ("phase_gap_unified_low", "u-low"),
        ("phase_gap_fixed_low", "f-low"),
        ("phase_gap_adaptive_low", "a-low"),
        ("phase_gap_unified_high", "u-high"),
        ("phase_gap_fixed_high", "f-high"),
        ("phase_gap_adaptive_high", "a-high"),
    ]

    for offset, (metric, label) in zip(np.linspace(-4.5, 4.5, len(metrics)), metrics):
        vals = [next(r for r in rows if r["k"] == k)[metric] for k in window_sizes]
        ax.bar(x + offset * width, vals, width, label=label)

    ax.set_xticks(x, window_sizes)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## Adaptive-gain heatmaps over unified

In [ ]:
def build_adaptive_gain_matrix(system_name):
    rows = [r for r in adapt_results if r["system"] == system_name]
    return np.array([
        [next(r for r in rows if r["k"] == k)["phase_gap_adaptive_low"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_low"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_adaptive_high"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_high"] for k in window_sizes],
        [next(r for r in rows if r["k"] == k)["phase_gap_adaptive_mixed"] - next(r for r in rows if r["k"] == k)["phase_gap_unified_mixed"] for k in window_sizes],
    ])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    M = build_adaptive_gain_matrix(system_name)
    im = ax.imshow(M, aspect="auto", origin="upper")
    ax.set_title(system_name)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])
    ax.set_xlabel("window size k")
    ax.set_ylabel("evaluation set")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="adaptive gain over unified")
plt.tight_layout()
plt.show()

## Overlap comparison curves with bootstrap intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in adapt_results if r["system"] == system_name]
    series = [
        ("zg_unified_mixed", "u-mixed"),
        ("zg_specialist_mixed", "s-mixed"),
        ("zg_fixed_mixed", "f-mixed"),
        ("zg_adaptive_mixed", "a-mixed"),
    ]

    for metric, label in series:
        means, lows, highs = [], [], []
        for k in window_sizes:
            row = next(r for r in rows if r["k"] == k)
            means.append(row[f"{metric}_mean"])
            low = row[f"{metric}_mean"] - row[f"{metric}_q025"]
            high = row[f"{metric}_q975"] - row[f"{metric}_mean"]
            lows.append(max(0.0, low))
            highs.append(max(0.0, high))
        ax.errorbar(window_sizes, means, yerr=np.vstack([np.array(lows), np.array(highs)]), marker="o", capsize=4, label=label)

    ax.set_title(f"{system_name}: zeta vs GUE overlap")
    ax.set_xlabel("window size k")
    ax.set_ylabel("overlap mean")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Learned α summaries

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in adapt_results if r["system"] == system_name]
    alpha_low = [next(r for r in rows if r["k"] == k)["alpha_low_mean"] for k in window_sizes]
    alpha_high = [next(r for r in rows if r["k"] == k)["alpha_high_mean"] for k in window_sizes]
    ax.plot(window_sizes, alpha_low, marker="o", label="alpha low")
    ax.plot(window_sizes, alpha_high, marker="o", label="alpha high")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("mean learned alpha")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Compact adaptive summary

In [ ]:
summary_rows = []
for system_name in ["entropy", "unevenness"]:
    for k in window_sizes:
        row = next(r for r in adapt_results if r["system"] == system_name and r["k"] == k)
        summary_rows.append({
            "system": system_name,
            "k": k,
            "adaptive_gain_low": float(row["phase_gap_adaptive_low"] - row["phase_gap_unified_low"]),
            "adaptive_gain_high": float(row["phase_gap_adaptive_high"] - row["phase_gap_unified_high"]),
            "adaptive_gain_mixed": float(row["phase_gap_adaptive_mixed"] - row["phase_gap_unified_mixed"]),
            "adaptive_minus_fixed_mixed": float(row["phase_gap_adaptive_mixed"] - row["phase_gap_fixed_mixed"]),
            "alpha_low_mean": float(row["alpha_low_mean"]),
            "alpha_high_mean": float(row["alpha_high_mean"]),
        })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "fixed_alpha": fixed_alpha,
    "n_boot": n_boot,
    "systems": list(systems.keys()),
    "summary_rows": summary_rows,
}
summary

## Notes

- If adaptive mixture improves over fixed mixture and unified, then the correction strength really is data-dependent.
- If learned α is small at large k, that supports the “unified core dominates at larger scales” picture.
- If learned α differs strongly between entropy and unevenness systems, then the two condition systems require different correction logic.

## Next directions

A natural next notebook could do one of these:

1. learn α from a richer gate network rather than a linear logistic gate
2. compare different gate inputs one by one
3. add surrogate-aware adaptive residuals
4. test adaptive gating across feature families